# TF-IDF (concatenated) + 1 hidden layer MLP

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:**

- same preprocessing as in the glove notebooks
- `TfidfVectorizer` from sklearn, fitted on all single sentences (O1, O2, H1, H2) of the training data
- the four tf-idf vectors are concatenated in fixed order O1, O2, H1, H2
- MLP: input -> hidden -> ReLU -> 2 outputs (one hidden layer less than `tfidf_bigger_mlp.ipynb`)

In [1]:
%pip install -q datasets lightning nltk python-dotenv scikit-learn scipy wandb


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import lightning as L
import nltk
import numpy as np
import torch
import torch.nn as nn
import wandb
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import WandbLogger
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import DataLoader, Dataset

nltk.download("punkt_tab", quiet=True)

True

In [3]:
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
print(f"environment loaded from: {dotenv_path or '<no .env found, using the defaults below>'}")

WANDB_BASE_URL = os.getenv("WANDB_BASE_URL", "https://api.wandb.ai")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "cmedi32-hochschule-luzern")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nlp-project-01-abductive-reasoning")

os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
config = {
    "dataset": "amarfurt/art-nlp-hs26",
    "tokenizer": "nltk.word_tokenize",
    "lowercase": True,
    "remove_punctuation": True,
    "remove_stopwords": False,
    "normalization": "none",
    "truncation": "none",
    "vectorizer": "sklearn.TfidfVectorizer",
    "vocabulary": "train sentences (O1, O2, H1, H2)",
    "example_representation": "concat",  # O1, O2, H1, H2
    "classifier": "mlp",  # linear -> relu -> linear
    "hidden_dim": 256,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "max_epochs": 100,
    "patience": 5,  # validation checks without improvement of loss/valid before stopping
    "seed": 42,
}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="tfidf-concatenate-small-mlp",
    config=config,
)

In [5]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")

train: 16801, valid: 1000, test: 1532


In [6]:
def preprocess(text):
    tokens = nltk.word_tokenize(text)
    return [token.lower() for token in tokens if any(character.isalnum() for character in token)]

In [7]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [8]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

In [9]:
# vocabulary: every single sentence (O1, O2, H1, H2) of the training data, already tokenized
all_sentences = []
for example in train_tokens:
    for field in TEXT_FIELDS:
        all_sentences.append(example[field])

# the sentences are already token lists, so the analyzer just hands them through unchanged
vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens)
vectorizer.fit(all_sentences)
print(f"vocabulary: {len(vectorizer.vocabulary_)} terms")

# one tf-idf matrix per field, glued side by side -> (examples, 4 * vocabulary)
X_train = sparse.hstack([
    vectorizer.transform(train_tokens["observation_1"]),
    vectorizer.transform(train_tokens["observation_2"]),
    vectorizer.transform(train_tokens["hypothesis_1"]),
    vectorizer.transform(train_tokens["hypothesis_2"]),
], format="csr")
y_train = np.array(train_tokens["target"])

X_valid = sparse.hstack([
    vectorizer.transform(valid_tokens["observation_1"]),
    vectorizer.transform(valid_tokens["observation_2"]),
    vectorizer.transform(valid_tokens["hypothesis_1"]),
    vectorizer.transform(valid_tokens["hypothesis_2"]),
], format="csr")
y_valid = np.array(valid_tokens["target"])

X_test = sparse.hstack([
    vectorizer.transform(test_tokens["observation_1"]),
    vectorizer.transform(test_tokens["observation_2"]),
    vectorizer.transform(test_tokens["hypothesis_1"]),
    vectorizer.transform(test_tokens["hypothesis_2"]),
], format="csr")
y_test = np.array(test_tokens["target"])

print(f"input dimension: {X_train.shape[1]}")

vocabulary: 16189 terms
input dimension: 64756


### TFIDF Vectorizer

### DataLoaders - Takes care of Training Batching and Loading

In [10]:
L.seed_everything(config["seed"])


class SparseDataset(Dataset):
    """Serves (features, label) pairs from a sparse tf-idf matrix.

    The full dense matrix would be examples x (4 * vocabulary) floats and not fit in memory,
    so a row is only densified when the DataLoader asks for it.
    """

    def __init__(self, X, y):
        self.X = X  # scipy csr matrix, one row per example
        self.y = torch.tensor(y, dtype=torch.long)  # 0 -> hypothesis_1, 1 -> hypothesis_2

    def __len__(self):
        # DataLoader uses this to know how many examples exist per epoch
        return self.X.shape[0]

    def __getitem__(self, index):
        # sparse row -> dense 1d float tensor of size 4 * vocabulary
        x = torch.tensor(self.X[index].toarray().ravel(), dtype=torch.float32)
        return x, self.y[index]


# DataLoader draws batch_size indices, calls __getitem__ for each and stacks the results into
# x: (batch_size, 4 * vocabulary) and y: (batch_size,). Only the training data is shuffled.
train_loader = DataLoader(SparseDataset(X_train, y_train), batch_size=config["batch_size"], shuffle=True)
valid_loader = DataLoader(SparseDataset(X_valid, y_valid), batch_size=config["batch_size"], shuffle=False)
test_loader = DataLoader(SparseDataset(X_test, y_test), batch_size=config["batch_size"], shuffle=False)

Seed set to 42


In [11]:
class MLP(L.LightningModule):

    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate):
        super().__init__()
        self.save_hyperparameters()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def reset_parameters(self):
        for layer in self.layers:
            if isinstance(layer, nn.Linear):
                layer.reset_parameters()

    def forward(self, x):
        return self.layers(x)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)

    def shared_step(self, batch, split):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        accuracy = (logits.argmax(dim=1) == y).float().mean()
        self.log(f"loss/{split}", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"accuracy/{split}", accuracy, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self.shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.shared_step(batch, "valid")

    def test_step(self, batch, batch_idx):
        return self.shared_step(batch, "test")

### Best Checkpoint + Upload Artifacts to Weights and Biases

In [12]:
WEIGHTS_DIR = Path("../../../models/project_1/tfidf_concatenate_small_mlp/weights")

model = MLP(
    input_dim=X_train.shape[1],
    hidden_dim=config["hidden_dim"],
    output_dim=2,
    learning_rate=config["learning_rate"],
)
model.reset_parameters()

early_stopping = EarlyStopping(monitor="loss/valid", mode="min", patience=config["patience"], verbose=True)
checkpoint = ModelCheckpoint(dirpath=WEIGHTS_DIR, filename="best-{epoch}", monitor="loss/valid", mode="min", save_top_k=1)

trainer = L.Trainer(
    max_epochs=config["max_epochs"],
    accelerator="auto",
    devices=1,
    logger=WandbLogger(),
    callbacks=[early_stopping, checkpoint],
)
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ layers  │ Sequential       │ 16.6 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 16.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 16.6 M                                                                                               
Total estimated model params size (MB): 66.313                                                                     
Modules in train mode: 5                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 
'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the 
`num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.

Metric loss/valid improved. New best score: 0.670
Monitored metric loss/valid did not improve in the last 5 records. Best score: 0.670. Signaling Trainer to stop.


In [13]:
trainer.validate(model, dataloaders=valid_loader, ckpt_path="best")
trainer.test(model, dataloaders=test_loader, ckpt_path="best")

run.summary.update({
    "accuracy/majority": np.bincount(y_test).max() / len(y_test),
    "best/loss_valid": checkpoint.best_model_score.item(),
    "best/checkpoint": Path(checkpoint.best_model_path).name,
    "early_stopping/stopped_epoch": early_stopping.stopped_epoch,
    "tfidf/vocabulary_size": len(vectorizer.vocabulary_),
    "tfidf/input_dim": X_train.shape[1],
})

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      accuracy/valid       │    0.5740000009536743     │
│        loss/valid         │    0.6695731282234192     │
└───────────────────────────┴───────────────────────────┘

Restoring states from the checkpoint path at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/models/project_1/tfidf_concatenate_small_mlp/weights/best-epoch=0.ckpt


Output()

/home/cedi/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       accuracy/test       │    0.5163185596466064     │
│         loss/test         │    0.7077447175979614     │
└───────────────────────────┴───────────────────────────┘

In [14]:
artifact = wandb.Artifact("tfidf-concatenate-small-mlp", type="model", metadata={"loss/valid": checkpoint.best_model_score.item()})
artifact.add_file(checkpoint.best_model_path)
run.log_artifact(artifact, aliases=["best"])
wandb.finish()

accuracy/test,▁
accuracy/train,▁▅▇███
accuracy/valid,▇█▄█▁▂▇
epoch,▁▁▂▂▃▃▅▅▆▆▇▇██
loss/test,▁
loss/train,█▆▃▁▁▁
loss/valid,▁▂▄▆▇█▁
trainer/global_step,▁▁▂▂▄▄▅▅▇▇████
accuracy/majority,0.50979
accuracy/test,0.51632
accuracy/train,0.99994
